In [13]:
# imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
import pandas as pd 
import numpy as np
from sklearn.decomposition import FastICA
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_fscore_support, classification_report

<h1>Dataset Preparation</h1>

<h3>Label aggregation</h3>

Convert the annotations from multiple annotators into training labels for each
time segment. You may use your insights from the exploration phase.

i. Which aggregation strategy did you use?
Expected Answer: A clear description of how annotations of shape [T, C, A] are processed into
labels of shape [T, C].

ii. Why is your aggregation strategy suitable for this classification task? What are its advantages
and potential limitations?
Expected Answer: A justification of the chosen strategy, including its advantages and potential
limitations compared to alternative approaches.

In [2]:
features_and_annotations = dict(
np.load("audio_features/000001.npz", allow_pickle=True)
)

features_and_annotations.keys()

dict_keys(['zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max', 'start_time', 'end_time', 'melspect_mean', 'melspect_std', 'melspect_min', 'melspect_max', 'mfcc_mean', 'mfcc_std', 'mfcc_min', 'mfcc_max', 'mfcc_d_mean', 'mfcc_d_std', 'mfcc_d_min', 'mfcc_d_max', 'mfcc_d2_mean', 'mfcc_d2_std', 'mfcc_d2_min', 'mfcc_d2_max', 'flux_mean', 'flux_std', 'flux_min', 'flux_max', 'flatness_mean', 'flatness_std', 'flatness_min', 'flatness_max', 'centroid_mean', 'centroid_std', 'centroid_min', 'centroid_max', 'bandwidth_mean', 'bandwidth_std', 'bandwidth_min', 'bandwidth_max', 'contrast_mean', 'contrast_std', 'contrast_min', 'contrast_max', 'rolloff_low_mean', 'rolloff_low_std', 'rolloff_low_min', 'rolloff_low_max', 'rolloff_high_mean', 'rolloff_high_std', 'rolloff_high_min', 'rolloff_high_max', 'energy_mean', 'energy_std', 'energy_min', 'energy_max', 'power_mean', 'power_std', 'power_min', 'power_max', 'annotations', 'is_own_recording', 'class_names', 'annotator_ids', 'target_classes', 'non_target_classes'

In [3]:
def extract_dataset(base_path, threshold=0.5):
    """    
    Returns:
        X: np.ndarray of shape [Total_Frames, Total_Features]
        y: np.ndarray of shape [Total_Frames, Total_Target_Classes]
    """
    npz_folder = os.path.join(base_path, 'audio_features') 
    metadata_path = os.path.join(base_path, 'metadata.csv')
    
    # 1. Parse metadata and extract a global, aligned list of target classes
    metadata = pd.read_csv(metadata_path).set_index('filename')
    global_classes = set()
    for target_str in metadata['target_classes'].dropna():
        for t in str(target_str).split(';'):
            global_classes.add(t.strip())
    global_classes = sorted(list(global_classes))
    class_to_idx = {cls: idx for idx, cls in enumerate(global_classes)}
    
    # 2. Define the exact time-varying features from the dataset specs
    feature_keys = [
        'zcr', 'melspect', 'mfcc', 'mfcc_d', 'mfcc_d2', 'flux', 
        'flatness', 'centroid', 'bandwidth', 'contrast', 
        'rolloff_low', 'rolloff_high', 'energy', 'power'
    ]
    suffixes = ['_mean', '_std', '_min', '_max']
    
    X_all = []
    y_all = []
    
    # 3. Process each feature file
    files = [f for f in os.listdir(npz_folder) if f.endswith('.npz')]
    for file in files:
        rec_id = file.replace('.npz', '.wav')
        if rec_id not in metadata.index: 
            continue
            
        target_str = str(metadata.loc[rec_id, 'target_classes'])
        file_targets = [t.strip() for t in target_str.split(';')]

        data = dict(np.load(os.path.join(npz_folder, file), allow_pickle=True))
        ann = data['annotations']      # Shape: [T, C, A]
        class_names = list(data['class_names'])
        T = ann.shape[0]               # Total time frames
        
        # --- FEATURE CONCATENATION (X) ---
        file_features = []
        for base_key in feature_keys:
            for suffix in suffixes:
                key = f"{base_key}{suffix}"
                if key in data:
                    feat = data[key]
                    # Convert 1D features [T] safely to 2D column vectors [T, 1]
                    if feat.ndim == 1:
                        feat = feat[:, np.newaxis]
                    file_features.append(feat)
                    
        X_file = np.concatenate(file_features, axis=1) # Shape: [T, Total_Features]
        
        # --- DYNAMIC LABEL AGGREGATION (y) ---
        # A. Calculate mean proportional overlap across any number of annotators (axis=2)
        mean_overlap = np.mean(ann, axis=2) # Shape transformation: [T, C, A] -> [T, C]
        
        # B. Apply agreement threshold (> 0.5) to produce consensus binary frames
        consensus = (mean_overlap > threshold).astype(int)
        
        # C. Align localized active target labels into the unified global class matrix
        y_file = np.zeros((T, len(global_classes)), dtype=int)
        for local_idx, name in enumerate(class_names):
            if name in file_targets and name in class_to_idx:
                global_idx = class_to_idx[name]
                y_file[:, global_idx] = consensus[:, local_idx]
                
        X_all.append(X_file)
        y_all.append(y_file)
        
    # Stack all files into final design matrices
    X = np.vstack(X_all) if X_all else np.empty((0, 0))
    y = np.vstack(y_all) if y_all else np.empty((0, 0))
    
    return X, y

path = './'
# X, y = extract_dataset(path)
# print("X shape:", X.shape)
# print("y shape:", y.shape)
# print(X[0, :10]) # example
# y[0]

Collapsing the Annotator Axis ($A$): For each 1-second time segment ($T$) and sound class ($C$), we compute the mathematical mean across the annotator dimension (axis 2). Because individual values in the array represent the exact proportional overlap ($0.0$ to $1.0$) of an event inside that window, taking the mean yields a continuous consensus score between $0.0$ and $1.0$.

Further threshold of 0.5 agrrement is applied to get 1 or 0

Because the dataset utilizes a overlapping hop size of 0.5 seconds , minor timing disagreements naturally yield fractional overlaps (e.g., $0.3$ or $0.7$). Averaging these values prior to binarization ensures that the final training label has integrity.

On one hand the threshold of 0.5 removes noice, but if there are 2 annotators it is unlikely that both of them annotate a quiet footstep and as it is strict ineqaulity both need to agree.

We loose information about the agrrement of the annotators.

In [4]:
# this code preserves classes like is_own_recording
def extract_dataset(base_path, files=None, threshold=0.5):
    """    
    Returns:
        X: np.ndarray of shape [Total_Frames, Total_Features]
        y: np.ndarray of shape [Total_Frames, Total_Target_Classes]
    """
    npz_folder = os.path.join(base_path, 'audio_features') 
    metadata_path = os.path.join(base_path, 'metadata.csv')
    
    # 1. Parse metadata and extract a global, aligned list of target classes
    metadata = pd.read_csv(metadata_path).set_index('filename')
    global_classes = set()
    for target_str in metadata['target_classes'].dropna():
        for t in str(target_str).split(';'):
            global_classes.add(t.strip())
    global_classes = sorted(list(global_classes))
    class_to_idx = {cls: idx for idx, cls in enumerate(global_classes)}
    
    # 2. Define standard time-varying acoustic keys
    feature_keys = [
        'zcr', 'melspect', 'mfcc', 'mfcc_d', 'mfcc_d2', 'flux', 
        'flatness', 'centroid', 'bandwidth', 'contrast', 
        'rolloff_low', 'rolloff_high', 'energy', 'power'
    ]
    suffixes = ['_mean', '_std', '_min', '_max']
    acoustic_keys = [f"{base}{suff}" for base in feature_keys for suff in suffixes]
    
    # --- META MAPPER FOR CATEGORICAL DATA ---
    # Dictionaries to dynamically map textual categories into unique numeric identifiers
    device_map = {}
    env_map = {}
    scene_map = {}
    placement_map = {}
    
    X_all = []
    y_all = []
    
    # 3. Process each feature file
    if not files:
        files = [f for f in os.listdir(npz_folder) if f.endswith('.npz')]
    for file in files:
        rec_id = file.replace('.npz', '.wav')
        if rec_id not in metadata.index: 
            continue
            
        target_str = str(metadata.loc[rec_id, 'target_classes'])
        file_targets = [t.strip() for t in target_str.split(';')]

        data = dict(np.load(os.path.join(npz_folder, file), allow_pickle=True))
        ann = data['annotations']      # Shape: [T, C, A]
        class_names = list(data['class_names'])
        T = ann.shape[0]               # Total time frames
        
        # --- FEATURE CONCATENATION (X) ---
        file_features = []
        
        # A. Append standard acoustic features
        for key in acoustic_keys:
            if key in data:
                feat = data[key]
                if feat.ndim == 1:
                    feat = feat[:, np.newaxis]
                file_features.append(feat)
                
        # B. Append 'is_own_recording' metadata 
        # (Takes the mean ownership presence across all annotators, broadcasts to [T, 1])
        if 'is_own_recording' in data:
            own_rec_ratio = np.mean(data['is_own_recording'].astype(float))
            own_rec_feat = np.full((T, 1), own_rec_ratio)
            file_features.append(own_rec_feat)
            
        # C. Process Categorical/String Non-acoustic Keys via Dynamic Encoding
        # We fetch the metadata label, convert it to a number, and broadcast it to [T, 1]
        categorical_specs = [
            ('recording_device', device_map),
            ('recording_environments', env_map),
            ('scene_description', scene_map),
            ('device_placement', placement_map)
        ]
        
        for meta_key, tracking_map in categorical_specs:
            if meta_key in data:
                # String representations can sometimes be arrays inside .npz files
                val_str = str(data[meta_key])
                if val_str not in tracking_map:
                    tracking_map[val_str] = len(tracking_map) # Assign next index integer
                
                encoded_val = tracking_map[val_str]
                meta_broadcasted = np.full((T, 1), encoded_val, dtype=float)
                file_features.append(meta_broadcasted)
                
        X_file = np.concatenate(file_features, axis=1) # Shape: [T, Total_Features]
        
        # --- DYNAMIC LABEL AGGREGATION (y) ---
        mean_overlap = np.mean(ann, axis=2)
        consensus = (mean_overlap > threshold).astype(int)
        
        y_file = np.zeros((T, len(global_classes)), dtype=int)
        for local_idx, name in enumerate(class_names):
            if name in file_targets and name in class_to_idx:
                global_idx = class_to_idx[name]
                y_file[:, global_idx] = consensus[:, local_idx]
                
        X_all.append(X_file)
        y_all.append(y_file)
        
    # Stack all files into final design matrices
    X = np.vstack(X_all) if X_all else np.empty((0, 0))
    y = np.vstack(y_all) if y_all else np.empty((0, 0))
    
    return X, y

path = './'
# # X, y = extract_dataset(path)
# print("X shape:", X.shape)
# print("y shape:", y.shape)

<h3>Data split</h3>

Describe how you split the data for model selection and performance evaluation. Remember: you will need to train your classifier, tune hyperparameters, and estimate final performance.

i. What are the potential causes of information leakage when performing the split and how can
you prevent it? Should audio segments from the same recording or recordings from the same
collector appear in multiple splits?
Expected Answer: A discussion of potential sources of information leakage (e.g., segments from
the same recording appearing in different splits) and a strategy to prevent it.

ii. How do you ensure that class distributions are consistent across the splits? Should the class
distribution be preserved across training, validation, and test sets?
Expected Answer: A justification of whether and why class distributions should be kept similar
across splits. A description of how this could be achieved. A table or visualization showing the
class distributions across the splits.


In [5]:
from torch import _test_autograd_multiple_dispatch


def split_dataset_stratified(base_path, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, threshold=0.5):
    """
    Groups data by recording file to prevent information leakage, while using 
    an iterative multi-label stratification algorithm to maintain balanced class distributions.
    """
    npz_folder = os.path.join(base_path, 'audio_features')
    metadata_path = os.path.join(base_path, 'metadata.csv')
    
    # 1. Parse metadata and map global target classes
    metadata = pd.read_csv(metadata_path).set_index('filename')
    global_classes = set()
    for target_str in metadata['target_classes'].dropna():
        for t in str(target_str).split(';'):
            global_classes.add(t.strip())
    global_classes = sorted(list(global_classes))
    class_to_idx = {cls: idx for idx, cls in enumerate(global_classes)}
    num_classes = len(global_classes)
    
    # 2. First pass: Extract true counts per file
    files = [f for f in os.listdir(npz_folder) if f.endswith('.npz')]
    file_profiles = []
    total_class_counts = np.zeros(num_classes)
    
    for file in files:
        rec_id = file.replace('.npz', '.wav')
        if rec_id not in metadata.index:
            continue
            
        target_str = str(metadata.loc[rec_id, 'target_classes'])
        file_targets = [t.strip() for t in target_str.split(';')]
        
        data = np.load(os.path.join(npz_folder, file), allow_pickle=True)
        ann = data['annotations']      # Shape: [T, C, A]
        class_names = list(data['class_names'])
        T = ann.shape[0]
        
        mean_overlap = np.mean(ann, axis=2)
        consensus = (mean_overlap > threshold).astype(int)
        
        y_file = np.zeros((T, num_classes), dtype=int)
        for local_idx, name in enumerate(class_names):
            if name in file_targets and name in class_to_idx:
                global_idx = class_to_idx[name]
                y_file[:, global_idx] = consensus[:, local_idx]
                
        class_counts = np.sum(y_file, axis=0)
        total_class_counts += class_counts
        
        file_profiles.append({
            'filename': file,
            'counts': class_counts,
            'total_frames': T
        })
        
    # Calculate the exact number of frames each class wants in each split
    target_ratios = {'train': train_ratio, 'val': val_ratio, 'test': test_ratio}
    class_split_targets = {
        split: total_class_counts * ratio for split, ratio in target_ratios.items()
    }
    
    # Track current accumulated counts per class per split
    split_counts = {
        'train': np.zeros(num_classes),
        'val': np.zeros(num_classes),
        'test': np.zeros(num_classes)
    }
    
    splits = {'train': [], 'val': [], 'test': []}
    
    # 3. Sort files based on their rarest active class overall
    # Files containing labels that are globally scarce get assigned first
    def get_rarity_score(profile):
        active_indices = np.where(profile['counts'] > 0)[0]
        if len(active_indices) == 0:
            return float('inf') # Push completely silent files to the back
        return np.min(total_class_counts[active_indices])

    file_profiles = sorted(file_profiles, key=get_rarity_score)
    
    # 4. Iterative Multilabel Allocation
    for profile in file_profiles:
        f_counts = profile['counts']
        
        # If the file contains no active targets, assign based entirely on split frame capacities
        if np.sum(f_counts) == 0:
            best_split = min(['train', 'val', 'test'], 
                             key=lambda s: np.sum(split_counts[s]) / target_ratios[s])
            splits[best_split].append(profile['filename'])
            split_counts[best_split] += f_counts
            continue
            
        # Compute "finding score" for each split option
        split_scores = {}
        for s in ['train', 'val', 'test']:
            # How much does this split still need each class present in this file?
            remaining_demand = class_split_targets[s] - split_counts[s]
            # Maximize demand coverage for active classes in this file
            active_demand = remaining_demand[f_counts > 0]
            split_scores[s] = np.sum(active_demand)
            
        # Find split with the maximum demand score
        max_score = max(split_scores.values())
        best_splits = [s for s, score in split_scores.items() if score == max_score]
        
        # If there's a tie in class demand, break it using the split with the lowest total size relative to its ratio
        if len(best_splits) > 1:
            best_split = min(best_splits, key=lambda s: np.sum(split_counts[s]) / target_ratios[s])
        else:
            best_split = best_splits[0]
            
        splits[best_split].append(profile['filename'])
        split_counts[best_split] += f_counts

    # 5. Print Distribution Report
    print("\n=== CLASS DISTRIBUTION REPORT ACROSS SPLITS ===")
    report_df = pd.DataFrame(split_counts, index=global_classes)
    
    # Calculate sum totals per column to establish accurate proportions
    train_total = report_df['train'].sum()
    val_total = report_df['val'].sum()
    test_total = report_df['test'].sum()
    
    report_df['train_%'] = (report_df['train'] / (train_total + 1e-6) * 100).round(2)
    report_df['val_%'] = (report_df['val'] / (val_total + 1e-6) * 100).round(2)
    report_df['test_%'] = (report_df['test'] / (test_total + 1e-6) * 100).round(2)
    
    print(report_df[['train', 'train_%', 'val', 'val_%', 'test', 'test_%']])

    train_distribution = dict(zip(global_classes, report_df['train'].values))
    val_distribution = dict(zip(global_classes, report_df['val'].values))
    test_distribution = dict(zip(global_classes, report_df['test'].values))
    
    return splits['train'], splits['val'], splits['test'], train_distribution, val_distribution, test_distribution
# --- RUNNING THE SPLIT ---
path = './'
train_files, val_files, test_files, train_distribution, val_distribution, test_distribution = split_dataset_stratified(path)


=== CLASS DISTRIBUTION REPORT ACROSS SPLITS ===
                              train  train_%     val  val_%    test  test_%
bell_ringing                 1964.0     1.97    59.0   0.34    62.0    0.35
coffee_machine               5576.0     5.58   269.0   1.55   300.0    1.72
cutlery_dishes               7557.0     7.56  1457.0   8.38  1424.0    8.14
door_open_close              4908.0     4.91   719.0   4.13   719.0    4.11
footsteps                   13999.0    14.01  2992.0  17.20  3010.0   17.21
keyboard_typing             11114.0    11.12  2403.0  13.81  2394.0   13.69
keychain                     5892.0     5.90  1058.0   6.08  1063.0    6.08
light_switch                  651.0     0.65     8.0   0.05    36.0    0.21
microwave                    8166.0     8.17  1749.0  10.05  1758.0   10.05
phone_ringing                7420.0     7.43  1589.0   9.13  1643.0    9.40
running_water               14719.0    14.73  3144.0  18.07  3165.0   18.10
toilet_flushing              5032.0    

In [6]:
# these are lists of filenames
print(f"\nTrain files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")
print(f"Test files: {len(test_files)}")


Train files: 2675
Validation files: 515
Test files: 466


In [7]:
# now get the actual data for each split
X_train, y_train = extract_dataset(path, files=train_files)
X_val, y_val = extract_dataset(path, files=val_files)
X_test, y_test = extract_dataset(path, files=test_files)

In [8]:
# shapes
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (124315, 965), y_train shape: (124315, 15)
X_val shape: (22932, 965), y_val shape: (22932, 15)
X_test shape: (20992, 965), y_test shape: (20992, 15)


The splits are made in a way such that one collector_id is found only in one split, meaning the creators of the files from validation and test set will be truly unseen.

For every recording file, we aggregate its annotations into consensus frames and sum the total active frames belonging to each class, pcreating a "class profile weight" vector for that file.

We sort all files based on the presence of the rarest classes (e.g., prioritizing files containing coffee_machine first).

We iteratively assign each file to whichever split (Train, Validation, or Test) has the highest relative deficit for that file's specific active classes based on target ratios (e.g., 70% / 15% / 15%).

<h3>Preprocessing</h3>

Did you apply any preprocessing techniques (e.g. data normalization, feature
selection) to the audio features? If so, explain which techniques you used and why they were necessary.
You may revisit and refine your preprocessing choices after selecting your model.
Expected Answer: A justified preprocessing pipeline, supported by experiments or reasoning about
the chosen classifier and feature properties.


In [9]:
scaler = StandardScaler()
ica = FastICA(n_components=50) 

# Scale features
X_train_scaled = scaler.fit_transform(X_train)
X_train_ica = ica.fit_transform(X_train_scaled)

X_val_scaled = scaler.transform(X_val)
X_val_ica = ica.transform(X_val_scaled)

X_test_scaled = scaler.transform(X_test)
X_test_ica = ica.transform(X_test_scaled)

print(f"Original Feature Count: {X_train.shape}")
print(f"Optimized Feature Count (ICA): {X_train_ica.shape}")
print(f"Train Shape: {X_train_ica.shape}, Val Shape: {X_val_ica.shape}, Test Shape: {X_test_ica.shape}")

Original Feature Count: (124315, 965)
Optimized Feature Count (ICA): (124315, 50)
Train Shape: (124315, 50), Val Shape: (22932, 50), Test Shape: (20992, 50)


Up to this point used ICA but when the performance is going to be terrible we should think about removing ICA.

Used a simple standard scaler as the features are way different in magnitude.

Afterwards CA was applied.


<h1>Evaluation</h1>

(a) Which evaluation criterion did you choose to compare hyperparameter settings and algorithms, and
why?

Expected Answer: Choose a metric that takes into account the dataset characteristics. You may use
your analysis from Task 3 to justify your choice.
(b) What is the baseline performance? What could be the best possible performance?

Hint: A simple baseline could, for example, generate predictions based on the empirical class frequencies (e.g., by sampling according to how often each class occurs in the data), without training a
classifier.
Expected Answer: Define and evaluate a simple baseline (i.e., without training a classifier). Report its
performance and discuss whether perfect performance is achievable given the dataset characteristics.

In [10]:
class FrequencyBaselineClassifier:
    def __init__(self, class_names):
        self.class_names = class_names
        self.class_probabilities = {}
        
    def fit(self, train_counts, total_train_frames):
        """
        Calculates the prior probability (frequency) of each class 
        based solely on its occurrence rate in the training set.
        """
        for cls, count in train_counts.items():
            # Probability that this sound event is active (1) in any given frame
            prob = count / total_train_frames
            self.class_probabilities[cls] = prob
            
    def predict(self, num_samples):
        """
        Generates dummy multi-label predictions for a given number of frames.
        X is ignored; predictions are sampled solely from class frequencies.
        """
        num_classes = len(self.class_names)
        # Initialize an empty matrix of shape [num_frames, num_classes]
        predictions = np.zeros((num_samples, num_classes), dtype=np.float64)
        
        for idx, cls in enumerate(self.class_names):
            prob = self.class_probabilities[cls]
            # Draw random samples from a binomial (Bernoulli) distribution based on the class frequency
            predictions[:, idx] = np.random.binomial(n=1, p=prob, size=num_samples)
            
        return predictions

# ==========================================
# 1. DEFINE TRAINING DISTRIBUTION (From your data)
# ==========================================
# Total frames can be derived from adding your total class counts or tracking the actual array length.
# Assuming the sum total of active timestamps divided by average active density: 
# Let's say your training split has a total of 100,210 independent 1-second frames.
TOTAL_TRAIN_FRAMES = X_train_ica.shape[0]

classes = list(train_distribution.keys())

# ==========================================
# 2. TRAIN & EVALUATE THE BASELINE MODEL
# ==========================================
# Instantiate the baseline model
baseline_model = FrequencyBaselineClassifier(class_names=classes)
baseline_model.fit(train_distribution, total_train_frames=TOTAL_TRAIN_FRAMES)

# (Sum of test instances or length of y_test frame rows)
TOTAL_TEST_FRAMES = X_test_ica.shape[0]
y_pred_test = baseline_model.predict(num_samples=TOTAL_TEST_FRAMES)

print(f"Generated baseline prediction matrix shape: {y_pred_test.shape}")
# print("Sample frame prediction (first frame):")
# for name, val in zip(classes, y_pred_test[0]):
#     print(f"  {name}: {val}")

Generated baseline prediction matrix shape: (20992, 15)


In [11]:



# =====================================================================
# 2. EVALUATE PERFORMANCE USING MACRO-AVERAGED METRICS
# =====================================================================
# Calculate independent Precision, Recall, and F1 per class, then average them (macro)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, 
    y_pred_test, 
    average='macro', 
    zero_division=0
)

print("=====================================================")
print("          GLOBAL METRICS (MACRO EVALUATION)          ")
print("=====================================================")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}")
print("=====================================================\n")

# Print out a clean class-by-class metric breakdown report
print("DETAILED CLASS-BY-CLASS PERFORMANCE BREAKDOWN:")
print(classification_report(
    y_test, 
    y_pred_test, 
    target_names=classes, 
    zero_division=0
))

          GLOBAL METRICS (MACRO EVALUATION)          
Macro Precision: 0.0567
Macro Recall:    0.0544
Macro F1-Score:  0.0523

DETAILED CLASS-BY-CLASS PERFORMANCE BREAKDOWN:
                            precision    recall  f1-score   support

              bell_ringing       0.00      0.02      0.01        62
            coffee_machine       0.01      0.02      0.01       300
            cutlery_dishes       0.08      0.07      0.08      1424
           door_open_close       0.04      0.04      0.04       719
                 footsteps       0.15      0.12      0.13      3010
           keyboard_typing       0.12      0.09      0.10      2394
                  keychain       0.05      0.05      0.05      1063
              light_switch       0.01      0.03      0.02        36
                 microwave       0.07      0.06      0.06      1758
             phone_ringing       0.08      0.06      0.07      1643
             running_water       0.16      0.12      0.14      3165
         

# TODO
explain why this is a good metric

<h1>Experiments</h1>

(a) Choose at least two classifiers from different model classes (Support Vector Machines, Nearest Neighbor Classifiers, Decision Trees, Neural Networks, etc). Systematically vary the most important hyperparameters and visualize the change in performance.
Expected Answer: For each classifier, analyze the most relevant hyperparameters and describe their
role in the classifier. Systematically vary these hyperparameters, visualize how they affect performance, and clearly explain how you selected the final values.

(b) After selecting appropriate hyperparameters, compare the final performance estimate of the two
chosen classifiers to the baseline performance established in 2.b.
Expected Answer: Both classifiers should outperform the baseline. Compare the classifiers with
respect to their performance and discuss their strengths and weaknesses. Explain why some classifiers
may be more suitable for the given dataset.

In [20]:
def train_and_tune_classifiers(X_train, y_train, X_val, y_val, class_names):
    """
    Trains, hyperparameter tunes, and evaluates the two most appropriate 
    classifiers for the multi-label audio frame dataset.
    """
    
    # =====================================================================
    # MODEL 1: MULTI-OUTPUT RANDOM FOREST
    # =====================================================================
    print("--- Training Multi-Output Random Forest ---")
    
    # Base estimator
    rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
    rf_multilabel = MultiOutputClassifier(rf_base)
    
    # Simple, fast parameter grid (prefix keys with 'estimator__' for MultiOutput)
    rf_param_grid = {
        'estimator__n_estimators': [50],
        'estimator__max_depth': [20]
    }
    
    # Pre-split validation strategy using PredefinedSplit could also be used, 
    # but a simple 3-fold CV on train tracks stability effectively.
    rf_grid = GridSearchCV(
        estimator=rf_multilabel,
        param_grid=rf_param_grid,
        scoring='f1_macro',
        cv=3,
        n_jobs=-1
    )
    
    print("Fitting Random Forest Grid...")
    rf_grid.fit(X_train, y_train)
    best_rf = rf_grid.best_estimator_
    print(f"Best Random Forest Parameters: {rf_grid.best_params_}")
    
    # Evaluate on Validation Set
    y_pred_rf_val = best_rf.predict(X_val)
    rf_macro_f1 = f1_score(y_val, y_pred_rf_val, average='macro', zero_division=0)
    print(f"Random Forest Validation Macro F1: {rf_macro_f1:.4f}\n")
    
    
    # =====================================================================
    # MODEL 2: MULTI-OUTPUT SUPPORT VECTOR MACHINE (LinearSVC)
    # =====================================================================
    print("--- Training Multi-Output Support Vector Machine ---")
    
    # LinearSVC is significantly faster on large frame datasets than SVC(kernel='rbf')
    svm_base = LinearSVC(random_state=42, class_weight='balanced', dual=False, max_iter=2000)
    svm_multilabel = MultiOutputClassifier(svm_base)
    
    # Simple regularization parameter grid
    svm_param_grid = {
        'estimator__C': [1.0]
    }
    
    svm_grid = GridSearchCV(
        estimator=svm_multilabel,
        param_grid=svm_param_grid,
        scoring='f1_macro',
        cv=3,
        n_jobs=-1
    )
    
    print("Fitting SVM Grid...")
    svm_grid.fit(X_train, y_train)
    best_svm = svm_grid.best_estimator_
    print(f"Best SVM Parameters: {svm_grid.best_params_}")
    
    # Evaluate on Validation Set
    y_pred_svm_val = best_svm.predict(X_val)
    svm_macro_f1 = f1_score(y_val, y_pred_svm_val, average='macro', zero_division=0)
    print(f"SVM Validation Macro F1: {svm_macro_f1:.4f}\n")
    
    
    # =====================================================================
    # COMPARATIVE EVALUATION SUMMARY
    # =====================================================================
    print("=====================================================")
    print("             FINAL VALIDATION REPORT                 ")
    print("=====================================================")
    print(f"Random Forest Macro F1: {rf_macro_f1:.4f}")
    print(f"Linear SVM Macro F1:    {svm_macro_f1:.4f}")
    print("=====================================================\n")
    
    # Choose the overall top-performing model to inspect class breakdowns
    winner_name = "Random Forest" if rf_macro_f1 > svm_macro_f1 else "Linear SVM"
    winner_model = best_rf if rf_macro_f1 > svm_macro_f1 else best_svm
    y_winner_pred = y_pred_rf_val if rf_macro_f1 > svm_macro_f1 else y_pred_svm_val
    
    print(f"Detailed Validation Breakdown for Top Model ({winner_name}):")
    print(classification_report(y_val, y_winner_pred, target_names=class_names, zero_division=0))
    
    return best_rf, best_svm

trained_rf, trained_svm = train_and_tune_classifiers(X_train_ica, y_train, X_val_ica, y_val, classes)

--- Training Multi-Output Random Forest ---
Fitting Random Forest Grid...
Best Random Forest Parameters: {'estimator__max_depth': 20, 'estimator__n_estimators': 50}
Random Forest Validation Macro F1: 0.3486

--- Training Multi-Output Support Vector Machine ---
Fitting SVM Grid...


c:\Users\pk\anaconda3\envs\ml-st\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
1 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\pk\anaconda3\envs\ml-st\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\pk\anaconda3\envs\ml-st\Lib\site-packages\sklearn\multioutput.py", line 547, in fit
    super().fit(X, Y, sample_weight=sample_weight, **fit_params)
  File "c:\Users\pk\anaconda3\envs\ml-st\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator,

Best SVM Parameters: {'estimator__C': 1.0}
SVM Validation Macro F1: 0.2690

             FINAL VALIDATION REPORT                 
Random Forest Macro F1: 0.3486
Linear SVM Macro F1:    0.2690

Detailed Validation Breakdown for Top Model (Random Forest):
                            precision    recall  f1-score   support

              bell_ringing       0.00      0.00      0.00        59
            coffee_machine       0.49      0.45      0.47       269
            cutlery_dishes       0.58      0.22      0.32      1457
           door_open_close       0.24      0.33      0.28       719
                 footsteps       0.39      0.40      0.40      2992
           keyboard_typing       0.72      0.36      0.48      2403
                  keychain       0.44      0.45      0.45      1058
              light_switch       0.02      0.50      0.04         8
                 microwave       0.84      0.42      0.56      1749
             phone_ringing       0.68      0.40      0.51      15

<h1>Case study and Reflections</h1>

(a) Find two interesting audio files that have not been used for training and qualitatively evaluate your
classifier’s predictions. Use the spectrogram and the sequence of predictions to visualize the classifier
output. Listen to the audios and inspect the corresponding predictions of the classifier. How well
does the classifier recognize the classes?
Expected Answer: A clear visualization of the predicted labels vs. the expected labels. Accompany
the visualization with text explaining both correct predictions and errors made by the classifier.

(b) Reflect on your overall classification results. What are typical failure cases? Which classes does the
classifier detect reliably, and which ones are difficult to detect?
Expected Answer: A discussion of both strengths and weaknesses of the classifier. Investigate class
ambiguity (e.g., using per-class confusion matrices) and use these to identify classes that are well
recognized and those that are difficult to detect.